# Proyecto 02 - Inteligencia Artificial
*Rodrigo Sebastián Ajmac Aroche - 22279*

## Configuración 

**Librerías**

**Dependencias**

## Funciones Auxiliares

### Colas Implementadas en el Laboratorio 3 

In [1]:
class Nodo:
    def __init__(self, estado, padre=None, costo_camino=0, heuristica=0):
        self.estado = estado
        self.padre = padre
        self.costo_camino = costo_camino
        self.heuristica = heuristica

    def costo_total(self):
        # f(n) = g(n) + h(n) para el algoritmo A*
        return self.costo_camino + self.heuristica

    def __repr__(self):
        return f"[{self.estado}]"

In [2]:
class ColaFIFO:
    def __init__(self):
        self.lista = []

    def empty(self):
        return len(self.lista) == 0

    def top(self):
        return self.lista[0] if not self.empty() else None

    def pop(self):
        return self.lista.pop(0) if not self.empty() else None

    def add(self, elemento):
        # FIFO: Se añade al final, el primero en llegar (índice 0) es el primero en salir.
        self.lista.append(elemento)
        return self.lista

In [3]:
class ColaLIFO:
    def __init__(self):
        self.lista = []

    def empty(self):
        return len(self.lista) == 0

    def top(self):
        return self.lista[0] if not self.empty() else None

    def pop(self):
        return self.lista.pop(0) if not self.empty() else None

    def add(self, elemento):
        # LIFO: Se añade al inicio (índice 0), el último en llegar es el primero en salir.
        self.lista.insert(0, elemento)
        return self.lista

In [4]:
class ColaPrioridad:
    def __init__(self, tipo_prioridad='costo'):
        self.lista = []
        # El tipo determina por qué atributo ordenamos: 'costo', 'heuristica', o 'a_star'
        self.tipo_prioridad = tipo_prioridad

    def empty(self):
        return len(self.lista) == 0

    def top(self):
        return self.lista[0] if not self.empty() else None

    def pop(self):
        return self.lista.pop(0) if not self.empty() else None

    def add(self, nodo):
        self.lista.append(nodo)
        # Ordenamos de menor a mayor dependiendo de la métrica escogida
        if self.tipo_prioridad == 'costo':
            self.lista.sort(key=lambda x: x.costo_camino)
        elif self.tipo_prioridad == 'heuristica':
            self.lista.sort(key=lambda x: x.heuristica)
        elif self.tipo_prioridad == 'a_star':
            self.lista.sort(key=lambda x: x.costo_total())
        return self.lista

### Funciones para Abrir .txt y Calcular Branching Factor

In [2]:
def leer_laberinto(ruta_archivo):
    """
    Lee un archivo de texto y lo convierte en una matriz bidimensional.
    Retorna la matriz del laberinto, una lista con los puntos de partida y 
    una lista con los puntos de salida.
    """
    laberinto = []
    puntos_partida = []
    puntos_salida = []

    with open(ruta_archivo, 'r') as archivo:
        for i, linea in enumerate(archivo):
            # Limpiamos los saltos de línea y espacios
            fila_texto = linea.strip()
            
            # Si la línea está vacía, la saltamos
            if not fila_texto:
                continue
                
            fila_int = []
            for j, caracter in enumerate(fila_texto):
                valor = int(caracter)
                fila_int.append(valor)
                
                # Identificar puntos clave
                if valor == 2:
                    puntos_partida.append((i, j))
                elif valor == 3:
                    puntos_salida.append((i, j))
                    
            laberinto.append(fila_int)

    return laberinto, puntos_partida, puntos_salida

In [1]:
def calcular_branching_factor(nodos_explorados, profundidad, tolerancia=0.0001):
    """
    Aproxima el factor de ramificación efectivo (b*) usando búsqueda binaria.
    - nodos_explorados: Total de nodos que el algoritmo sacó de la cola.
    - profundidad: Longitud del camino encontrado (costo de la solución).
    - tolerancia: Qué tan preciso queremos que sea el cálculo.
    """
    if profundidad == 0:
        return 0.0
    if profundidad == 1:
        return float(nodos_explorados)

    # Rango inicial para la búsqueda binaria
    low = 0.0
    high = float(nodos_explorados) 

    while high - low > tolerancia:
        mid = (low + high) / 2.0
        
        # Calcular los nodos generados teóricos con el valor 'mid'
        # Usamos la fórmula de la serie geométrica para optimizar: (b^(d+1) - b) / (b - 1)
        if mid == 1.0:
            calculado = profundidad
        else:
            calculado = (mid ** (profundidad + 1) - mid) / (mid - 1)

        # Ajustar los límites de la búsqueda binaria
        if calculado < nodos_explorados:
            low = mid
        else:
            high = mid

    # Retornar el punto medio como la mejor aproximación
    return (low + high) / 2.0